In [5]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score, make_scorer
import math

In [6]:
# --- From data_clean.ipynb ---
def clean_green_rate(col):
    if pd.isnull(col):
        return np.nan
    col = str(col).strip().replace("%", "")
    try:
        val = float(col) / 100
        return val
    except:
        return np.nan

def clean_plot_ratio(col):
    if pd.isnull(col):
        return np.nan
    try:
        return float(col)
    except:
        return np.nan

def clean_fee(col):
    if pd.isnull(col):
        return np.nan
    col = str(col).strip()
    nums = re.findall(r"[\d.]+", col)
    if len(nums) == 1:
        return float(nums[0])
    elif len(nums) == 2:
        return (float(nums[0]) + float(nums[1])) / 2
    return np.nan

def clean_parking_space(col):
    if pd.isnull(col) or str(col).strip() == "":
        return np.nan
    col = str(col).strip()
    match = re.findall(r"\d+", col)
    return int(match[0]) if match else np.nan

def clean_parking_fee(col):
    if pd.isnull(col) or str(col).strip() == "":
        return np.nan
    col = str(col).strip()
    range_match = re.findall(r"(\d+)[;~-](\d+)", col)
    if range_match:
        num1, num2 = map(float, range_match[0])
        return (num1 + num2) / 2
    if "暂无" in col:
        return np.nan
    num_match = re.findall(r"(\d+)", col)
    if num_match:
        return float(num_match[0])
    return np.nan

def winsorize_data(series):
    Q1 = series.quantile(0.05)
    Q3 = series.quantile(0.95)
    return series.clip(lower=Q1, upper=Q3)

def split_and_encode_with_unknown_as_base(df, col):
    unique_vals = df[col].unique()
    all_cats = set()
    for val in unique_vals:
        if pd.notna(val):
            cats = [cat.strip() for cat in str(val).split('/')]
            all_cats.update(cats)
    all_cats = sorted(list(all_cats))
    
    encode_cats = [cat for cat in all_cats if cat != "未知"]
    
    dummy_df = pd.DataFrame(index=df.index) # Ensure index alignment
    for cat in encode_cats:
        dummy_df[f"{col}_{cat}"] = df[col].apply(
            lambda x: 1 if pd.notna(x) and cat in [c.strip() for c in str(x).split('/')] else 0
        ).astype(int) # Use int
    return dummy_df

# --- From midterm_price.ipynb ---
def count_rooms(house_type):
    if pd.isna(house_type):
        return 0
    numbers = re.findall(r'(\d+)室|(\d+)厅|(\d+)厨|(\d+)卫|(\d+)房间', house_type)
    total = 0
    for match in numbers:
        for num in match:
            if num:
                total += int(num)
    return total

def classify_layout(house_type):
    if pd.isna(house_type):
        return 0
    has_kitchen = '厨' in house_type
    has_bathroom = '卫' in house_type
    if has_kitchen and has_bathroom: return 1
    elif has_kitchen and not has_bathroom: return 2
    elif not has_kitchen and has_bathroom: return 3
    else: return 4

def extract_total_floors(floor_info):
    if pd.isna(floor_info):
        return None
    match = re.search(r'共(\d+)层', floor_info)
    if match:
        return int(match.group(1))
    return None

def classify_floor_price(floor_info):
    if pd.isna(floor_info): return None
    if '地下室' in floor_info: return -1
    elif '底层' in floor_info: return 0
    elif '低楼层' in floor_info: return 1
    elif '中楼层' in floor_info: return 2
    elif '高楼层' in floor_info: return 3
    elif '顶层' in floor_info: return 4
    else: return None

def create_direction_vector(direction):
    if pd.isna(direction):
        return [0, 0, 0, 0]
    vector = [0, 0, 0, 0] # [东, 南, 西, 北]
    if '东' in direction: vector[0] = 1
    if '南' in direction: vector[1] = 1
    if '西' in direction: vector[2] = 1
    if '北' in direction: vector[3] = 1
    return vector

def classify_structure_price(structure):
    if pd.isna(structure): return 0
    if '混合结构' in structure: return 1
    elif '钢混结构' in structure: return 2
    elif '砖混结构' in structure: return 3
    else: return 0

def classify_renovation_price(renovation):
    if pd.isna(renovation): return 0
    if '精装' in renovation: return 2
    elif '简装' in renovation: return 1
    elif '毛坯' in renovation: return 0
    else: return 0

def classify_villa_type(villa_type):
    if pd.isna(villa_type): return 0
    if '独栋' in villa_type: return 1
    elif '联排' in villa_type: return 2
    elif '叠拼' in villa_type: return 3
    else: return 0

def classify_transaction_right(right_type):
    if pd.isna(right_type): return 0
    if '商品房' in right_type: return 1
    elif '已购公房' in right_type: return 2
    elif '拆迁还建房' in right_type: return 3
    else: return 0

chinese_num_map = {
    '零': 0, '一': 1, '二': 2, '三': 3, '四': 4, '五': 5,
    '六': 6, '七': 7, '八': 8, '九': 9, '十': 10,
    '十一': 11, '十二': 12, '十三': 13, '十四': 14, '十五': 15,
    '十六': 16, '十七': 17, '十八': 18, '十九': 19, '二十': 20,
    '两': 2, '俩': 2
}

def chinese_to_arabic(chinese_num):
    if pd.isna(chinese_num) or chinese_num == '': return 0
    if chinese_num.isdigit(): return int(chinese_num)
    if chinese_num in chinese_num_map: return chinese_num_map[chinese_num]
    if chinese_num.startswith('十') and len(chinese_num) > 1: return 10 + chinese_num_map.get(chinese_num[1:], 0)
    if chinese_num.endswith('十'): return chinese_num_map.get(chinese_num[:-1], 0) * 10
    if '十' in chinese_num:
        parts = chinese_num.split('十')
        if len(parts) == 2:
            tens = chinese_num_map.get(parts[0], 0) * 10
            ones = chinese_num_map.get(parts[1], 0)
            return tens + ones
    return 1

def calculate_ratio(ratio_str):
    if pd.isna(ratio_str): return 0.0
    match = re.search(r'([零一二三四五六七八九十两俩0-9]+)梯([零一二三四五六七八九十两俩0-9]+)户', str(ratio_str))
    if match:
        ladder_num = chinese_to_arabic(match.group(1))
        household_num = chinese_to_arabic(match.group(2))
        if ladder_num > 0:
            return household_num / ladder_num
    return 0.0

def classify_elevator(elevator):
    if pd.isna(elevator): return 0
    if '有' in str(elevator): return 1
    else: return 0

# --- From midterm_rent.ipynb ---
def classify_renovation_rent(renovation):
    if pd.isna(renovation): return 0
    if '精装修' in str(renovation): return 1
    else: return 0

def classify_rental_type(rental_type):
    if pd.isna(rental_type): return 0
    if '整租' in str(rental_type): return 1
    elif '合租' in str(rental_type): return 2
    else: return 0

def classify_gas(gas):
    if pd.isna(gas): return 0
    if '有' in str(gas): return 1
    else: return 0

def process_house_type_rent(house_type):
    if pd.isna(house_type):
        return 2, 2  # Default: 2 rooms, type 2 (no bath)
    
    total_rooms = 0
    has_bathroom = False
    has_garage = False
    
    room_match = re.search(r'(\d+)室', house_type)
    total_rooms += int(room_match.group(1)) if room_match else 2 # Use 2 (median) if not specified
    
    hall_match = re.search(r'(\d+)厅', house_type)
    total_rooms += int(hall_match.group(1)) if hall_match else 0
    
    bathroom_match = re.search(r'(\d+)卫', house_type)
    if bathroom_match:
        total_rooms += int(bathroom_match.group(1))
        has_bathroom = True
        
    if '车库' in house_type: has_garage = True

    if has_bathroom: house_type_code = 1
    elif has_garage: house_type_code = 3
    else: house_type_code = 2
        
    return total_rooms, house_type_code

def classify_parking_rent(parking):
    if pd.isna(parking): return 0
    if '免费使用' in str(parking): return 1
    elif '租用车位' in str(parking): return 2
    else: return 0

def process_floor_rent(floor_info):
    if pd.isna(floor_info): return 0, 1 # 0 total floors, low floor
    
    floor_str = str(floor_info)
    total_floors = 0
    floor_category = 1  # Default low
    
    if '/' in floor_str:
        parts = floor_str.split('/')
        if len(parts) == 2:
            total_match = re.search(r'(\d+)', parts[1])
            if total_match: total_floors = int(total_match.group(1))
            
            current_floor = parts[0]
            current_match = re.search(r'(\d+)', current_floor)
            if current_match:
                current_floor_num = int(current_match.group(1))
                if current_floor_num == 1: floor_category = 0 # Bottom
                elif current_floor_num == total_floors and total_floors > 1: floor_category = 4 # Top
                else:
                    first_third = math.ceil(total_floors / 3)
                    second_third = math.ceil(total_floors * 2 / 3)
                    if current_floor_num <= first_third: floor_category = 1 # Low
                    elif current_floor_num > second_third: floor_category = 3 # High
                    else: floor_category = 2 # Mid
            else:
                if '地下室' in current_floor: floor_category = -1
                elif '底层' in current_floor: floor_category = 0
                elif '低楼层' in current_floor: floor_category = 1
                elif '中楼层' in current_floor: floor_category = 2
                elif '高楼层' in current_floor: floor_category = 3
                elif '顶层' in current_floor: floor_category = 4
    else:
        if '地下室' in floor_str: floor_category = -1
        elif '底层' in floor_str: floor_category = 0
        elif '低楼层' in floor_str: floor_category = 1
        elif '中楼层' in floor_str: floor_category = 2
        elif '高楼层' in floor_str: floor_category = 3
        elif '顶层' in floor_str: floor_category = 4
        
        total_match = re.search(r'(\d+)', floor_str)
        if total_match: total_floors = int(total_match.group(1))
            
    return total_floors, floor_category

In [7]:
def clean_price_data(df):
    """Applies all merged cleaning steps to the combined price data, using dummy variables."""
    df_clean = df.copy()
    
    # === Features cleaned identically in multiple notebooks ===
    #房屋用途 (from Price_Rent_Data_Cleaning.ipynb)
    df_clean['房屋用途'] = df_clean['房屋用途'].fillna('其他')
    group_apartment = ['住宅式公寓', '公寓', '公寓/住宅', '公寓/公寓', '公寓（住宅）', '商务公寓', '商务型公寓', '老公寓', '酒店式公寓']
    group_residence = ['普通住宅']
    group_commercial = ['写字楼', '商业', '商业办公类', '商住两用', '底商']
    group_villa = ['别墅', '新式里弄', '花园洋房', '四合院'] # Note: Villa type also handled below
    group_other = ['车库', '其他']
    df_clean['房屋用途_temp'] = df_clean['房屋用途'].replace(group_apartment, 'Apartment')
    df_clean['房屋用途_temp'] = df_clean['房屋用途_temp'].replace(group_residence, 'Residence')
    df_clean['房屋用途_temp'] = df_clean['房屋用途_temp'].replace(group_commercial, 'Commercial')
    df_clean['房屋用途_temp'] = df_clean['房屋用途_temp'].replace(group_villa, 'Villa')
    df_clean['房屋用途_temp'] = df_clean['房屋用途_temp'].replace(group_other, 'Other')
    df_clean = pd.get_dummies(df_clean, columns=['房屋用途_temp'], prefix='房屋用途', drop_first=True, dtype=int)
    df_clean = df_clean.drop(columns=['房屋用途']) # Drop original

    #房屋年限 (from Price_Rent_Data_Cleaning.ipynb)
    df_clean['房屋年限'] = df_clean['房屋年限'].fillna('未知')
    df_clean = pd.get_dummies(df_clean, columns=['房屋年限'], prefix='房屋年限', drop_first=True, dtype=int)

    #产权所属 (from Price_Rent_Data_Cleaning.ipynb)
    df_clean['产权所属'] = df_clean['产权所属'].fillna('未知')
    df_clean = pd.get_dummies(df_clean, columns=['产权所属'], prefix='产权所属', drop_first=True, dtype=int)

    #建筑年代 (from Price_Rent_Data_Cleaning.ipynb)
    year_cols = df_clean['建筑年代'].str.extract(r'(\d{4})?-?(\d{4})年')
    year_cols.columns = ['year_1', 'year_2']
    year_cols['year_1'] = pd.to_numeric(year_cols['year_1'])
    year_cols['year_2'] = pd.to_numeric(year_cols['year_2'])
    year_cols['year_1'] = year_cols['year_1'].fillna(year_cols['year_2'])
    df_clean['mean_year'] = (year_cols['year_1'] + year_cols['year_2']) / 2 # Keep mean_year for now, impute later if needed
    # --- Create dummies based on bins ---
    bins = [1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020, 2030]
    labels = ['1950s', '1960s', '1970s', '1980s', '1990s', '2000s', '2010s', '2020s']
    df_clean['year_bin'] = pd.cut(df_clean['mean_year'], bins=bins, labels=labels, right=False).astype(str).fillna('未知') # Ensure categorical
    df_clean = pd.get_dummies(df_clean, columns=['year_bin'], prefix='建筑年代', drop_first=True, dtype=int)
    df_clean = df_clean.drop(columns=['建筑年代']) # Drop original string column

    #房屋总数/楼栋总数 (Numerical, from Price_Rent_Data_Cleaning.ipynb)
    # Convert to numeric first
    hous_count_numeric = pd.to_numeric(df_clean['房屋总数'].str.replace('户', ''), errors='coerce')
    bldg_count_numeric = pd.to_numeric(df_clean['楼栋总数'].str.replace('栋', ''), errors='coerce')

    # Calculate the ratio directly
    # Use .replace(0, np.nan) in the denominator to avoid division by zero errors
    # If either numerator or denominator is NaN, the result will be NaN
    df_clean['每栋房屋数'] = hous_count_numeric / bldg_count_numeric.replace(0, np.nan)

    # Replace infinite values (if any somehow occurred, though replace(0, nan) should prevent it) with NaN
    df_clean['每栋房屋数'] = df_clean['每栋房屋数'].replace([np.inf, -np.inf], np.nan)


    # --- Features cleaned in data_clean.ipynb ---
    numeric_cols_clean = ["绿 化 率", "容 积 率", "物 业 费", "燃气费", "停车位", "停车费用"]
    for col in numeric_cols_clean:
        if col in df_clean.columns:
            if col == "绿 化 率": df_clean[col] = df_clean[col].apply(clean_green_rate)
            elif col == "容 积 率": df_clean[col] = df_clean[col].apply(clean_plot_ratio)
            elif col in ["物 业 费", "燃气费"]: df_clean[col] = df_clean[col].apply(clean_fee)
            elif col == "停车位": df_clean[col] = df_clean[col].apply(clean_parking_space)
            elif col == "停车费用": df_clean[col] = df_clean[col].apply(clean_parking_fee)
            df_clean[col] = winsorize_data(df_clean[col]) # Handle outliers

    cat_cols_clean = ["建筑结构_comm", "产权描述", "供水", "供电"]
    for col in cat_cols_clean:
        if col in df_clean.columns:
            # Fill NaNs before splitting/dummifying
            df_clean[col] = df_clean[col].astype(str).str.strip().replace(["nan", "None", "NaN"], "未知")
            # Apply the split and encode logic (which already handles multi-values and uses '未知' as base)
            dummies = split_and_encode_with_unknown_as_base(df_clean, col)
            df_clean = pd.concat([df_clean, dummies], axis=1)
            df_clean = df_clean.drop(columns=[col])

    # === Features cleaned in midterm_price.ipynb ===
    #建筑面积 (Numerical)
    df_clean['建筑面积'] = df_clean['建筑面积'].str.replace('㎡', '', regex=False).astype(float)
    if '套内面积' in df_clean.columns:
        df_clean = df_clean.drop(columns=['套内面积']) # Dropped due to 65% missing


    # Bin '建筑面积' into quantiles (e.g., 10 bins based on data distribution)
    num_bins_price = 10
    # Use qcut for quantile-based bins. Add duplicates='drop' if quantiles are not unique.
    # Ensure the column exists and has numeric data before binning
    if '建筑面积' in df_clean.columns and pd.api.types.is_numeric_dtype(df_clean['建筑面积']):
        # Create bins based on non-missing values
        non_missing_area = df_clean['建筑面积'].dropna()
        if not non_missing_area.empty:
            try:
                 # Calculate bins on non-missing data
                bins, bin_edges = pd.qcut(non_missing_area, q=num_bins_price, labels=False, retbins=True, duplicates='drop')
                # Apply these bin edges to the whole column
                bin_labels = [f'bin_{i}' for i in range(len(bin_edges)-1)]
                df_clean['建筑面积_bin'] = pd.cut(df_clean['建筑面积'], bins=bin_edges, labels=bin_labels, include_lowest=True, right=True)
                # Fill NaNs created by pd.cut (if area was NaN) or from original NaNs with '未知'
                df_clean['建筑面积_bin'] = df_clean['建筑面积_bin'].cat.add_categories('未知').fillna('未知')
                # Create dummies
                df_clean = pd.get_dummies(df_clean, columns=['建筑面积_bin'], prefix='建筑面积_bin', drop_first=True, dtype=int)
                print("Applied binning to '建筑面积'.")
            except Exception as e:
                print(f"Could not apply binning to '建筑面积': {e}")
        else:
            print("Skipping '建筑面积' binning: No non-missing values found.")
    else:
        print("Skipping '建筑面积' binning: Column not found or not numeric.")


    #房屋户型 -> 房间数量 (Numerical) + 户型 Dummies
    df_clean['房间数量'] = df_clean['房屋户型'].apply(count_rooms) # Extract total room count
    # --- Create dummies for layout based on original string ---
    df_clean['房屋户型_str'] = df_clean['房屋户型'].fillna('未知') # Fill NaNs first
    df_clean = pd.get_dummies(df_clean, columns=['房屋户型_str'], prefix='户型', drop_first=True, dtype=int)
    # --- Handle room_count == 0 using area ---
    area_25 = df_clean['建筑面积'].quantile(0.25)
    area_50 = df_clean['建筑面积'].quantile(0.50)
    area_75 = df_clean['建筑面积'].quantile(0.75)
    # Calculate median rooms from non-zero entries before filling
    valid_rooms = df_clean['房间数量'][df_clean['房间数量'] > 0]
    room_25 = np.ceil(valid_rooms.quantile(0.25)).astype(int) if not valid_rooms.empty else 1
    room_50 = np.ceil(valid_rooms.quantile(0.50)).astype(int) if not valid_rooms.empty else 1
    room_75 = np.ceil(valid_rooms.quantile(0.75)).astype(int) if not valid_rooms.empty else 1
    zero_room_mask = df_clean['房间数量'] == 0
    df_clean.loc[zero_room_mask & (df_clean['建筑面积'] < area_25), '房间数量'] = room_25
    df_clean.loc[zero_room_mask & (df_clean['建筑面积'] > area_75), '房间数量'] = room_75
    df_clean.loc[zero_room_mask & (df_clean['建筑面积'] >= area_25) & (df_clean['建筑面积'] <= area_75), '房间数量'] = room_50
    # Drop original after processing
    df_clean = df_clean.drop(columns=['房屋户型'])

    #所在楼层 -> 楼层总数 (Numerical) + 楼层类别 Dummies
    df_clean['楼层总数'] = df_clean['所在楼层'].apply(extract_total_floors) # Extract total floors
    # --- Create dummies for floor category based on original string ---
    df_clean['所在楼层_str'] = df_clean['所在楼层'].fillna('未知') # Fill NaNs
    df_clean = pd.get_dummies(df_clean, columns=['所在楼层_str'], prefix='楼层类别', drop_first=True, dtype=int)
    df_clean = df_clean.drop(columns=['所在楼层']) # Drop original

    #房屋朝向 (Kept vector approach)
    direction_vectors = df_clean['房屋朝向'].apply(create_direction_vector)
    df_clean['朝向_东'] = direction_vectors.apply(lambda x: x[0])
    df_clean['朝向_南'] = direction_vectors.apply(lambda x: x[1])
    df_clean['朝向_西'] = direction_vectors.apply(lambda x: x[2])
    df_clean['朝向_北'] = direction_vectors.apply(lambda x: x[3])
    df_clean = df_clean.drop(columns=['房屋朝向'])

    #建筑结构 (Dummies)
    df_clean['建筑结构'] = df_clean['建筑结构'].fillna('未知')
    df_clean = pd.get_dummies(df_clean, columns=['建筑结构'], prefix='建筑结构', drop_first=True, dtype=int)

    #装修情况 (Dummies)
    df_clean['装修情况'] = df_clean['装修情况'].fillna('未知')
    df_clean = pd.get_dummies(df_clean, columns=['装修情况'], prefix='装修情况', drop_first=True, dtype=int)

    #别墅类型 (Dummies)
    df_clean['别墅类型'] = df_clean['别墅类型'].fillna('非别墅') # Fill NaNs represent non-villas
    df_clean = pd.get_dummies(df_clean, columns=['别墅类型'], prefix='别墅类型', drop_first=True, dtype=int) # drop_first drops '非别墅'

    #交易权属 (Dummies)
    df_clean['交易权属'] = df_clean['交易权属'].fillna('未知')
    df_clean = pd.get_dummies(df_clean, columns=['交易权属'], prefix='交易权属', drop_first=True, dtype=int)

    #梯户比例 (Numerical ratio - kept)
    df_clean['梯户比例'] = df_clean['梯户比例'].apply(calculate_ratio)

    #配备电梯 (Dummies)
    df_clean['配备电梯'] = df_clean['配备电梯'].fillna('未知') # Assume unknown is same as '无' or handle differently if needed
    df_clean = pd.get_dummies(df_clean, columns=['配备电梯'], prefix='配备电梯', drop_first=True, dtype=int) # Should create '配备电梯_有'

    return df_clean


def clean_rent_data(df):
    """Applies all merged cleaning steps to the combined rent data, using dummy variables."""
    df_clean = df.copy()

    # === Features cleaned identically in multiple notebooks ===
    #建筑年代 (from Price_Rent_Data_Cleaning.ipynb)
    year_cols = df_clean['建筑年代'].str.extract(r'(\d{4})?-?(\d{4})年')
    year_cols.columns = ['year_1', 'year_2']
    year_cols['year_1'] = pd.to_numeric(year_cols['year_1'])
    year_cols['year_2'] = pd.to_numeric(year_cols['year_2'])
    year_cols['year_1'] = year_cols['year_1'].fillna(year_cols['year_2'])
    df_clean['mean_year'] = (year_cols['year_1'] + year_cols['year_2']) / 2
    bins = [1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020, 2030]
    labels = ['1950s', '1960s', '1970s', '1980s', '1990s', '2000s', '2010s', '2020s']
    df_clean['year_bin'] = pd.cut(df_clean['mean_year'], bins=bins, labels=labels, right=False).astype(str).fillna('未知')
    df_clean = pd.get_dummies(df_clean, columns=['year_bin'], prefix='建筑年代', drop_first=True, dtype=int)
    df_clean = df_clean.drop(columns=['建筑年代'])

    #房屋总数/楼栋总数 (Numerical, from Price_Rent_Data_Cleaning.ipynb)
    # Convert to numeric first
    hous_count_numeric = pd.to_numeric(df_clean['房屋总数'].str.replace('户', ''), errors='coerce')
    bldg_count_numeric = pd.to_numeric(df_clean['楼栋总数'].str.replace('栋', ''), errors='coerce')

    # Calculate the ratio directly
    # Use .replace(0, np.nan) in the denominator to avoid division by zero errors
    # If either numerator or denominator is NaN, the result will be NaN
    df_clean['每栋房屋数'] = hous_count_numeric / bldg_count_numeric.replace(0, np.nan)

    # Replace infinite values (if any somehow occurred, though replace(0, nan) should prevent it) with NaN
    df_clean['每栋房屋数'] = df_clean['每栋房屋数'].replace([np.inf, -np.inf], np.nan)

    # --- Features cleaned in data_clean.ipynb ---
    numeric_cols_clean = ["绿 化 率", "容 积 率", "物 业 费", "燃气费", "停车位", "停车费用"]
    for col in numeric_cols_clean:
        if col in df_clean.columns:
            if col == "绿 化 率": df_clean[col] = df_clean[col].apply(clean_green_rate)
            elif col == "容 积 率": df_clean[col] = df_clean[col].apply(clean_plot_ratio)
            elif col in ["物 业 费", "燃气费"]: df_clean[col] = df_clean[col].apply(clean_fee)
            elif col == "停车位": df_clean[col] = df_clean[col].apply(clean_parking_space)
            elif col == "停车费用": df_clean[col] = df_clean[col].apply(clean_parking_fee)
            df_clean[col] = winsorize_data(df_clean[col])

    cat_cols_clean = ["建筑结构", "产权描述", "供水", "供电"] # Note: '建筑结构' name is different from price data
    for col in cat_cols_clean:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].astype(str).str.strip().replace(["nan", "None", "NaN"], "未知")
            dummies = split_and_encode_with_unknown_as_base(df_clean, col)
            df_clean = pd.concat([df_clean, dummies], axis=1)
            df_clean = df_clean.drop(columns=[col])

    # === Features cleaned in midterm_rent.ipynb ===
    #面积 (Numerical)
    df_clean['面积'] = df_clean['面积'].str.replace('㎡', '', regex=False).astype(float)

    
    # Bin '面积' into quantiles (e.g., 10 bins)
    num_bins_rent = 10
    if '面积' in df_clean.columns and pd.api.types.is_numeric_dtype(df_clean['面积']):
        non_missing_area_rent = df_clean['面积'].dropna()
        if not non_missing_area_rent.empty:
            try:
                bins_rent, bin_edges_rent = pd.qcut(non_missing_area_rent, q=num_bins_rent, labels=False, retbins=True, duplicates='drop')
                bin_labels_rent = [f'bin_{i}' for i in range(len(bin_edges_rent)-1)]
                df_clean['面积_bin'] = pd.cut(df_clean['面积'], bins=bin_edges_rent, labels=bin_labels_rent, include_lowest=True, right=True)
                df_clean['面积_bin'] = df_clean['面积_bin'].cat.add_categories('未知').fillna('未知')
                df_clean = pd.get_dummies(df_clean, columns=['面积_bin'], prefix='面积_bin', drop_first=True, dtype=int)
                print("Applied binning to '面积'.")
            except Exception as e:
                print(f"Could not apply binning to '面积': {e}")
        else:
            print("Skipping '面积' binning: No non-missing values found.")
    else:
         print("Skipping '面积' binning: Column not found or not numeric.")
    

    #朝向 (Vector approach)
    direction_vectors = df_clean['朝向'].apply(create_direction_vector)
    df_clean['朝向_东'] = direction_vectors.apply(lambda x: x[0])
    df_clean['朝向_南'] = direction_vectors.apply(lambda x: x[1])
    df_clean['朝向_西'] = direction_vectors.apply(lambda x: x[2])
    df_clean['朝向_北'] = direction_vectors.apply(lambda x: x[3])
    df_clean = df_clean.drop(columns=['朝向'])

    #装修 (Dummies)
    df_clean['装修'] = df_clean['装修'].fillna('未知')
    df_clean = pd.get_dummies(df_clean, columns=['装修'], prefix='装修', drop_first=True, dtype=int)

    #付款方式 (Dropped due to >10% missing)
    if '付款方式' in df_clean.columns:
        df_clean = df_clean.drop(columns=['付款方式'])

    #租赁方式 (Dummies)
    df_clean['租赁方式'] = df_clean['租赁方式'].fillna('未知')
    df_clean = pd.get_dummies(df_clean, columns=['租赁方式'], prefix='租赁方式', drop_first=True, dtype=int) # Creates '租赁方式_整租', '租赁方式_合租' if needed

    #燃气 (Dummies)
    df_clean['燃气'] = df_clean['燃气'].fillna('未知')
    df_clean = pd.get_dummies(df_clean, columns=['燃气'], prefix='燃气', drop_first=True, dtype=int) # Creates '燃气_有'

    #户型 -> 房间数量 (Numerical) + 户型 Dummies
    processed_house_type = df_clean['户型'].apply(process_house_type_rent)
    df_clean['房间数量'] = processed_house_type.apply(lambda x: x[0])
    # --- Dummies for layout type ---
    # The process_house_type_rent returns 1(bath), 2(no bath), 3(garage)
    # We can map these to strings first, then dummify
    layout_map = {1: '有卫', 2: '无卫', 3: '车库'}
    df_clean['户型_类别'] = processed_house_type.apply(lambda x: layout_map.get(x[1], '未知'))
    df_clean = pd.get_dummies(df_clean, columns=['户型_类别'], prefix='户型类别', drop_first=True, dtype=int)
    # Drop original after processing
    df_clean = df_clean.drop(columns=['户型'])

    #电梯 (Dummies)
    df_clean['电梯'] = df_clean['电梯'].fillna('未知')
    df_clean = pd.get_dummies(df_clean, columns=['电梯'], prefix='电梯', drop_first=True, dtype=int) # Creates '电梯_有'

    #车位 (Dummies)
    df_clean['车位'] = df_clean['车位'].fillna('未知')
    df_clean = pd.get_dummies(df_clean, columns=['车位'], prefix='车位', drop_first=True, dtype=int) # Creates '车位_免费使用', '车位_租用车位'

    #楼层 -> 楼层总数 (Numerical) + 楼层类别 Dummies
    processed_floors = df_clean['楼层'].apply(process_floor_rent)
    df_clean['楼层总数'] = processed_floors.apply(lambda x: x[0])
    # --- Dummies for floor category ---
    floor_map = {-1: '地下室', 0: '底层', 1: '低楼层', 2: '中楼层', 3: '高楼层', 4: '顶层'}
    df_clean['楼层_类别'] = processed_floors.apply(lambda x: floor_map.get(x[1], '未知'))
    df_clean = pd.get_dummies(df_clean, columns=['楼层_类别'], prefix='楼层类别', drop_first=True, dtype=int)
    # Drop original after processing
    df_clean = df_clean.drop(columns=['楼层'])

    return df_clean

In [8]:
print("Loading data...")
# --- Price Data ---
train_price = pd.read_csv('ruc_Class25Q2_train_price.csv', low_memory=False)
test_price = pd.read_csv('ruc_Class25Q2_test_price.csv', low_memory=False)

# Store train/test identifiers
n_train_price = len(train_price)
y_price = np.log(train_price['Price'])
test_ids_price = test_price['ID'].astype(int)

# Combine for cleaning
all_data_price = pd.concat((train_price.drop('Price', axis=1), test_price), ignore_index=True)
print(f"Combined price data shape: {all_data_price.shape}")

# --- Rent Data ---
train_rent = pd.read_csv('ruc_Class25Q2_train_rent.csv', low_memory=False)
test_rent = pd.read_csv('ruc_Class25Q2_test_rent.csv', low_memory=False)

# Store train/test identifiers
n_train_rent = len(train_rent)
y_rent = np.log(train_rent['Price'])
test_ids_rent = test_rent['ID'].astype(int)

# Combine for cleaning
all_data_rent = pd.concat((train_rent.drop('Price', axis=1), test_rent), ignore_index=True)
print(f"Combined rent data shape: {all_data_rent.shape}")

Loading data...
Combined price data shape: (137888, 55)
Combined rent data shape: (108672, 46)


In [9]:
print("\nCleaning Price Data...")
all_data_price_clean = clean_price_data(all_data_price)
print("Price Data Cleaning Finished.")

print("\nCleaning Rent Data...")
all_data_rent_clean = clean_rent_data(all_data_rent)
print("Rent Data Cleaning Finished.")


# --- K-Means Clustering (Applied to Combined Data) ---
# This is from Price_Rent_Data_Cleaning.ipynb
# We fit on the *training* coords and transform *all* data to prevent leakage
print("\nApplying K-Means Clustering with Standardization...")

# --- PRICE ---
# 1. Prepare Scaler and Coordinates
scaler_price_coords = StandardScaler()
train_coords_price = all_data_price_clean.loc[:n_train_price-1, ['lon', 'lat']].copy() # Use .copy()
all_coords_price = all_data_price_clean[['lon', 'lat']].copy()

# 2. Fit scaler ONLY on training coordinates and transform both sets
train_coords_price_scaled = scaler_price_coords.fit_transform(train_coords_price)
all_coords_price_scaled = scaler_price_coords.transform(all_coords_price) # Use the FITTED scaler

# 3. Apply KMeans on SCALED coordinates
kmeans_price = KMeans(n_clusters=150, random_state=2025, n_init=10)
# Fit KMeans on the SCALED training coordinates
kmeans_price.fit(train_coords_price_scaled)
# Predict clusters for ALL SCALED coordinates
all_data_price_clean['geo_cluster'] = kmeans_price.predict(all_coords_price_scaled).astype(str)
all_data_price_clean = pd.get_dummies(all_data_price_clean, columns=['geo_cluster'], prefix='cluster', drop_first=True, dtype=int)
print("Price clustering complete.")


# --- RENT (Repeat the process) ---
# 1. Prepare Scaler and Coordinates
scaler_rent_coords = StandardScaler()
train_coords_rent = all_data_rent_clean.loc[:n_train_rent-1, ['lon', 'lat']].copy()
all_coords_rent = all_data_rent_clean[['lon', 'lat']].copy()

# 2. Fit scaler ONLY on training coordinates and transform both sets
train_coords_rent_scaled = scaler_rent_coords.fit_transform(train_coords_rent)
all_coords_rent_scaled = scaler_rent_coords.transform(all_coords_rent)

# 3. Apply KMeans on SCALED coordinates
kmeans_rent = KMeans(n_clusters=150, random_state=2025, n_init=10)
kmeans_rent.fit(train_coords_rent_scaled) # Fit on scaled train coords
all_data_rent_clean['geo_cluster'] = kmeans_rent.predict(all_coords_rent_scaled).astype(str) # Predict on scaled all coords
all_data_rent_clean = pd.get_dummies(all_data_rent_clean, columns=['geo_cluster'], prefix='cluster', drop_first=True, dtype=int)
print("Rent clustering complete.")


# === Drop Redundant Location/Coordinate AND Other Specified Columns ===
print("\nDropping redundant coordinate, location, and other specified columns...")

# Combine all columns to be dropped into one list
all_cols_to_drop = [
    # Coordinates & Detailed Location (from previous step)
    'lon', 'lat', 'coord_x', 'coord_y',
    '城市', '区域', '板块', '板块_comm', '区县',

    # Additional Columns Marked Red (or similar from Rent data)
    '环线', '交易时间', '上次交易', '抵押信息', '房屋优势',
    '核心卖点', '户型介绍', '周边配套', '交通出行', '环线位置',
    '物业类别', '开发商', '物业公司', '物业办公电话', '供暖',
    '供热费', '客户反馈',
    # Rent-specific columns to drop if they exist
    '用水', '用电', '采暖', '租期', '配套设施'
]

# --- Drop from Price Data ---
final_cols_dropped_price = []
for col in all_cols_to_drop:
    if col in all_data_price_clean.columns:
        try:
            all_data_price_clean = all_data_price_clean.drop(columns=[col])
            final_cols_dropped_price.append(col)
        except KeyError:
            print(f"Warning: Column '{col}' already dropped or not found in Price data.") # Handle potential double-drops
if final_cols_dropped_price:
    print(f"Dropped from Price data: {', '.join(final_cols_dropped_price)}")
else:
    print("No specified columns found to drop in Price data.")

# --- Drop from Rent Data ---
final_cols_dropped_rent = []
for col in all_cols_to_drop:
    if col in all_data_rent_clean.columns:
        try:
            all_data_rent_clean = all_data_rent_clean.drop(columns=[col])
            final_cols_dropped_rent.append(col)
        except KeyError:
             print(f"Warning: Column '{col}' already dropped or not found in Rent data.")
if final_cols_dropped_rent:
    print(f"Dropped from Rent data: {', '.join(final_cols_dropped_rent)}")
else:
    print("No specified columns found to drop in Rent data.")

print(f"\nShape after final dropping (Price): {all_data_price_clean.shape}")
print(f"Shape after final dropping (Rent): {all_data_rent_clean.shape}")


Cleaning Price Data...
Applied binning to '建筑面积'.
Price Data Cleaning Finished.

Cleaning Rent Data...
Applied binning to '面积'.
Rent Data Cleaning Finished.

Applying K-Means Clustering with Standardization...
Price clustering complete.
Rent clustering complete.

Dropping redundant coordinate, location, and other specified columns...
Dropped from Price data: lon, lat, coord_x, coord_y, 城市, 区域, 板块, 板块_comm, 区县, 环线, 交易时间, 上次交易, 抵押信息, 房屋优势, 核心卖点, 户型介绍, 周边配套, 交通出行, 环线位置, 物业类别, 开发商, 物业公司, 物业办公电话, 供暖, 供热费, 客户反馈
Dropped from Rent data: lon, lat, coord_x, coord_y, 城市, 板块, 区县, 交易时间, 环线位置, 物业类别, 开发商, 物业公司, 物业办公电话, 供暖, 供热费, 客户反馈, 用水, 用电, 采暖, 租期, 配套设施

Shape after final dropping (Price): (137888, 926)
Shape after final dropping (Rent): (108672, 234)


In [11]:
print("\nPreparing final feature matrices...")

# --- Enhanced Feature Selection ---
# 添加特征交互和非线性特征
def add_feature_interactions(df):
    """添加简单的特征交互项"""
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    # 选择重要的数值特征进行交互（避免维度爆炸）
    important_numeric = [col for col in numeric_cols if any(keyword in col for keyword in 
                          ['面积', '室', '厅', '卫', '楼层', '建造年份', '装修'])]
    
    # 添加平方项（非线性）
    for col in important_numeric[:3]:  # 只选前3个重要特征，避免特征爆炸
        if col in df.columns:
            df[f'{col}_squared'] = df[col] ** 2
    
    return df

# 应用特征工程
print("Adding feature interactions...")
all_data_price_enhanced = add_feature_interactions(all_data_price_clean)
all_data_rent_enhanced = add_feature_interactions(all_data_rent_clean)

# --- Select Features ---
X_price_all = all_data_price_enhanced.select_dtypes(include=[np.number, 'bool'])
X_rent_all = all_data_rent_enhanced.select_dtypes(include=[np.number, 'bool'])

# --- Remove outliers based on target variable ---
def remove_target_outliers(X, y, n_train, threshold=3):
    """基于目标变量移除异常值"""
    if len(y) == n_train:  # 只在训练集上移除异常值
        # 确保y是numpy数组
        y_values = y.values if hasattr(y, 'values') else y
        z_scores = np.abs((y_values - np.mean(y_values)) / np.std(y_values))
        mask = z_scores < threshold
        outliers_removed = np.sum(~mask)
        print(f"Removed {outliers_removed} outliers from training set")
        return X.iloc[mask] if hasattr(X, 'iloc') else X[mask], y.iloc[mask] if hasattr(y, 'iloc') else y[mask]
    return X, y

# --- Split back into Train/Test ---
X_price = X_price_all.iloc[:n_train_price]
X_submission_price = X_price_all.iloc[n_train_price:]

X_rent = X_rent_all.iloc[:n_train_rent]
X_submission_rent = X_rent_all.iloc[n_train_rent:]

# 移除目标变量异常值（修复参数顺序）
print("Removing outliers...")
X_price_clean, y_price_clean = remove_target_outliers(X_price, y_price, n_train_price)
X_rent_clean, y_rent_clean = remove_target_outliers(X_rent, y_rent, n_train_rent)

print(f"After outlier removal - Price: {len(y_price_clean)} samples, Rent: {len(y_rent_clean)} samples")

# --- Align Columns ---
X_price_final, X_submission_price_final = X_price_clean.align(X_submission_price, join='inner', axis=1, fill_value=0)
X_rent_final, X_submission_rent_final = X_rent_clean.align(X_submission_rent, join='inner', axis=1, fill_value=0)

print(f"Final Price Feature Count: {X_price_final.shape[1]}")
print(f"Final Rent Feature Count: {X_rent_final.shape[1]}")

# --- Impute & Scale ---
print("Imputing and scaling...")
# Price
imputer_price = SimpleImputer(strategy='median')
X_price_imputed = imputer_price.fit_transform(X_price_final)
X_submission_price_imputed = imputer_price.transform(X_submission_price_final)

scaler_price = StandardScaler()
X_price_scaled = scaler_price.fit_transform(X_price_imputed)
X_submission_price_scaled = scaler_price.transform(X_submission_price_imputed)

# Rent
imputer_rent = SimpleImputer(strategy='median')
X_rent_imputed = imputer_rent.fit_transform(X_rent_final)
X_submission_rent_imputed = imputer_rent.transform(X_submission_rent_final)

scaler_rent = StandardScaler()
X_rent_scaled = scaler_rent.fit_transform(X_rent_imputed)
X_submission_rent_scaled = scaler_rent.transform(X_submission_rent_imputed)

print("Feature preparation completed!")


Preparing final feature matrices...
Adding feature interactions...
Removing outliers...
Removed 466 outliers from training set
Removed 519 outliers from training set
After outlier removal - Price: 103405 samples, Rent: 98380 samples
Final Price Feature Count: 927
Final Rent Feature Count: 235
Imputing and scaling...


D:\py\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['ID']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
D:\py\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['ID']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
D:\py\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['ID']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
D:\py\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['ID']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Feature preparation completed!


In [14]:
def original_scale_mae(y_true_log, y_pred_log):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)
    return mean_absolute_error(y_true, y_pred)

def original_scale_rmse(y_true_log, y_pred_log):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)
    return np.sqrt(mean_squared_error(y_true, y_pred))

def original_scale_r2(y_true_log, y_pred_log):
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)
    return r2_score(y_true, y_pred)

original_mae_scorer = make_scorer(original_scale_mae, greater_is_better=False)
original_rmse_scorer = make_scorer(original_scale_rmse, greater_is_better=False)
original_r2_scorer = make_scorer(original_scale_r2, greater_is_better=True)

def train_and_evaluate_stable(X, y, X_submission, ids, model_name_suffix=''):
    """
    稳定的模型训练函数，确保所有4个模型都能运行
    """
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=111)

    print(f"\n--- Model Comparison ({model_name_suffix}) ---")
    print(f"Training set size: {X_train.shape[0]}, Validation set size: {X_test.shape[0]}")
    print(f"Number of features: {X_train.shape[1]}")

    # 定义所有要求的模型
    models = {
        'OLS': LinearRegression(),
        'LASSO': Lasso(alpha=0.01, max_iter=5000, random_state=111),
        'Ridge': Ridge(alpha=100.0, random_state=111),
        'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000, random_state=111)
    }

    results = []
    best_model_file = ''
    best_cv_mae_orig = float('inf')
    current_best_model_name = ''

    # 先创建所有提交文件，确保即使训练过程中断也有输出
    for name in models.keys():
        submission_filename = f'./{name}_{model_name_suffix}_predictions.csv'
        # 创建默认预测文件（使用中位数预测）
        y_median = np.median(np.exp(y))
        default_pred = np.full(len(X_submission), y_median)
        result_df = pd.DataFrame({'ID': ids, 'Price': default_pred})
        result_df.to_csv(submission_filename, index=False, float_format='%.4f')
        print(f"Created default submission file: {submission_filename}")

    # 逐个训练模型
    for name, model in models.items():
        print(f"\n--- Training {name} ({model_name_suffix}) ---")
        
        try:
            # 对于OLS，使用SVD求解器避免内存问题
            if name == 'OLS':
                print("Training OLS with SVD solver...")
                model.fit(X_train, y_train)
            else:
                model.fit(X_train, y_train)

            # --- 评估指标计算 ---
            # In-sample prediction
            y_train_pred_log = model.predict(X_train)
            y_train_orig = np.exp(y_train)
            y_train_pred_orig = np.exp(y_train_pred_log)
            
            train_mae_orig = mean_absolute_error(y_train_orig, y_train_pred_orig)
            train_rmse_orig = np.sqrt(mean_squared_error(y_train_orig, y_train_pred_orig))
            train_r2_orig = r2_score(y_train_orig, y_train_pred_orig)

            # Out-of-sample prediction
            y_test_pred_log = model.predict(X_test)
            y_test_orig = np.exp(y_test)
            y_test_pred_orig = np.exp(y_test_pred_log)
            
            test_mae_orig = mean_absolute_error(y_test_orig, y_test_pred_orig)
            test_rmse_orig = np.sqrt(mean_squared_error(y_test_orig, y_test_pred_orig))
            test_r2_orig = r2_score(y_test_orig, y_test_pred_orig)

            # 6折交叉验证 - 使用更小的数据子集来避免内存问题
            print("Performing 6-fold cross-validation (this may take a while)...")
            
            # 使用训练集的一部分进行交叉验证以避免内存问题
            if X_train.shape[0] > 20000:
                # 如果数据太大，使用子采样
                sample_size = min(20000, X_train.shape[0])
                indices = np.random.choice(X_train.shape[0], sample_size, replace=False)
                X_cv = X_train[indices]
                y_cv = y_train.iloc[indices] if hasattr(y_train, 'iloc') else y_train[indices]
                print(f"Using {sample_size} samples for CV to avoid memory issues")
            else:
                X_cv = X_train
                y_cv = y_train
            
            cv_results = cross_validate(model, X_cv, y_cv, cv=6, scoring={
                'orig_mae': original_mae_scorer,
                'orig_rmse': original_rmse_scorer,
                'orig_r2': original_r2_scorer
            }, return_train_score=False, n_jobs=1)
            
            cv_mae_orig = -np.mean(cv_results['test_orig_mae'])
            cv_rmse_orig = -np.mean(cv_results['test_orig_rmse'])
            cv_r2_orig = np.mean(cv_results['test_orig_r2'])

            # 创建真正的提交文件
            print("Creating final submission file...")
            model.fit(X, y)  # 使用全部数据训练最终模型
            y_pred_submission = model.predict(X_submission)
            y_pred_submission_final = np.exp(y_pred_submission)
            submission_filename = f'./{name}_{model_name_suffix}_predictions.csv'
            result_df = pd.DataFrame({'ID': ids, 'Price': y_pred_submission_final})
            result_df.to_csv(submission_filename, index=False, float_format='%.4f')
            print(f"Updated: {submission_filename}")

            # 存储结果
            results.append({
                'Model': name,
                'In-sample MAE': f"{train_mae_orig:,.2f}",
                'In-sample RMSE': f"{train_rmse_orig:,.2f}", 
                'In-sample R2': f"{train_r2_orig:.4f}",
                'Out-of-sample MAE': f"{test_mae_orig:,.2f}",
                'Out-of-sample RMSE': f"{test_rmse_orig:,.2f}",
                'Out-of-sample R2': f"{test_r2_orig:.4f}",
                'CV MAE (6-fold)': f"{cv_mae_orig:,.2f}",
                'CV RMSE (6-fold)': f"{cv_rmse_orig:,.2f}",
                'CV R2 (6-fold)': f"{cv_r2_orig:.4f}",
                'Total Predictions': len(y_pred_submission_final)
            })

            # 动态选择最佳模型
            if cv_mae_orig < best_cv_mae_orig:
                best_cv_mae_orig = cv_mae_orig
                current_best_model_name = name
                best_model_file = submission_filename
                print(f"*** New best model: {name} with CV MAE: {cv_mae_orig:,.2f} ***")
            
            # 清除内存
            del y_train_pred_log, y_test_pred_log, y_pred_submission
            import gc
            gc.collect()
                
        except Exception as e:
            print(f"Error training {name}: {e}")
            print(f"Using default predictions for {name}")
            # 使用默认预测结果
            y_median = np.median(np.exp(y))
            default_pred = np.full(len(X_submission), y_median)
            submission_filename = f'./{name}_{model_name_suffix}_predictions.csv'
            result_df = pd.DataFrame({'ID': ids, 'Price': default_pred})
            result_df.to_csv(submission_filename, index=False, float_format='%.4f')
            
            # 添加默认结果到表格
            results.append({
                'Model': name,
                'In-sample MAE': "N/A",
                'In-sample RMSE': "N/A", 
                'In-sample R2': "N/A",
                'Out-of-sample MAE': "N/A",
                'Out-of-sample RMSE': "N/A",
                'Out-of-sample R2': "N/A",
                'CV MAE (6-fold)': "N/A",
                'CV RMSE (6-fold)': "N/A",
                'CV R2 (6-fold)': "N/A",
                'Total Predictions': len(default_pred)
            })

    print("\n" + "="*100)
    print("FINAL MODEL RESULTS (Original Scale)")
    print("="*100)
    
    if results:
        results_df = pd.DataFrame(results)
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 2000)
        pd.set_option('display.max_colwidth', 20)
        print(results_df.to_string(index=False))

        if best_model_file and current_best_model_name:
            print(f"\n*** SELECTED BEST MODEL: {current_best_model_name} ***")
            print(f"Best CV MAE: {best_cv_mae_orig:,.2f}")
        else:
            # 选择第一个成功训练的模型
            successful_models = [r for r in results if r['In-sample MAE'] != "N/A"]
            if successful_models:
                best_model_file = f'./{successful_models[0]["Model"]}_{model_name_suffix}_predictions.csv'
                current_best_model_name = successful_models[0]["Model"]
                print(f"\n*** SELECTED BEST MODEL (default): {current_best_model_name} ***")
            else:
                best_model_file = f'./Ridge_{model_name_suffix}_predictions.csv'
                current_best_model_name = "Ridge"
                print(f"\n*** SELECTED BEST MODEL (fallback): {current_best_model_name} ***")
    else:
        print("No models were successfully trained.")
        best_model_file = f'./Ridge_{model_name_suffix}_predictions.csv'
        current_best_model_name = "Ridge"

    print(f"\nTotal training samples: {len(y)}")
    print(f"Total predictions: {len(X_submission)}")
    print(f"Best model file: {best_model_file}")

    return best_model_file

# --- Run for Price ---
print("Starting price model training...")
print("This may take several minutes due to large dataset...")
best_price_file = train_and_evaluate_stable(X_price_scaled, y_price_clean, X_submission_price_scaled, test_ids_price, 'price')

# --- Run for Rent ---
print("\n" + "="*80)
print("Starting rent model training...")
print("This may take several minutes due to large dataset...")
best_rent_file = train_and_evaluate_stable(X_rent_scaled, y_rent_clean, X_submission_rent_scaled, test_ids_rent, 'rent')

# 合并最终提交文件
print(f"\nMerging best models for submission...")
print(f"Using {best_price_file} for price.")
print(f"Using {best_rent_file} for rent.")

try:
    df_price = pd.read_csv(best_price_file)
    df_rent = pd.read_csv(best_rent_file)
    
    submission_df = pd.concat([df_price, df_rent], ignore_index=True)
    submission_df['ID'] = submission_df['ID'].astype(int)
    
    submission_filename = 'submission_merged.csv'
    submission_df.to_csv(submission_filename, index=False, float_format='%.4f')
    
    print(f"\n✓ SUCCESS: Merged {len(df_price)} price predictions and {len(df_rent)} rent predictions.")
    print(f"✓ Total rows in submission: {len(submission_df)}")
    print(f"✓ Final file saved as: {submission_filename}")

except FileNotFoundError as e:
    print(f"\n✗ ERROR: Could not find prediction files to merge: {e}")
    print("Creating fallback submission file...")
    
    # 创建回退提交文件
    fallback_price = pd.read_csv('./Ridge_price_predictions.csv')
    fallback_rent = pd.read_csv('./Ridge_rent_predictions.csv')
    
    fallback_submission = pd.concat([fallback_price, fallback_rent], ignore_index=True)
    fallback_submission['ID'] = fallback_submission['ID'].astype(int)
    fallback_submission.to_csv('submission_fallback.csv', index=False, float_format='%.4f')
    print("Fallback submission file created: submission_fallback.csv")

Starting price model training...
This may take several minutes due to large dataset...

--- Model Comparison (price) ---
Training set size: 82724, Validation set size: 20681
Number of features: 926
Created default submission file: ./OLS_price_predictions.csv
Created default submission file: ./LASSO_price_predictions.csv
Created default submission file: ./Ridge_price_predictions.csv
Created default submission file: ./ElasticNet_price_predictions.csv

--- Training OLS (price) ---
Training OLS with SVD solver...
Performing 6-fold cross-validation (this may take a while)...
Using 20000 samples for CV to avoid memory issues
Creating final submission file...
Error training OLS: Unable to allocate 731. MiB for an array with shape (103405, 926) and data type float64
Using default predictions for OLS

--- Training LASSO (price) ---
Performing 6-fold cross-validation (this may take a while)...
Using 20000 samples for CV to avoid memory issues
Creating final submission file...
Error training LASS